In [5]:
import pandas as pd

recommended = pd.read_csv(
    "../data/raw/works_recommended.csv"
)

sanctioned = pd.read_csv(
    "../data/raw/works_sanctioned.csv"
)

completed = pd.read_csv(
    "../data/raw/works_completed.csv"
)

expenditure = pd.read_csv(
    "../data/raw/expenditure.csv"
)

allocated = pd.read_csv(
    "../data/raw/allocated_limit.csv"
)

C:\Users\thean\AppData\Local\Temp\ipykernel_9100\2389109442.py:3: DtypeWarning: Columns (0,9) have mixed types. Specify dtype option on import or set low_memory=False.
  recommended = pd.read_csv(
C:\Users\thean\AppData\Local\Temp\ipykernel_9100\2389109442.py:7: DtypeWarning: Columns (0,10) have mixed types. Specify dtype option on import or set low_memory=False.
  sanctioned = pd.read_csv(


In [6]:
def clean(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ","_")
    )

    df = df.drop_duplicates()

    return df

recommended = clean(recommended)
sanctioned = clean(sanctioned)
completed = clean(completed)

In [7]:
print(recommended.columns)
print(sanctioned.columns)
print(completed.columns)

Index(['sr._no.', 'work_category', 'work', 'state', 'ida',
       'hon'ble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'recommended_amount___(_₹_)', 'sanction_date'],
      dtype='object')
Index(['sr._no.', 'work_category', 'work', 'state', 'ida',
       'hon'ble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'sanction_date', 'sanction_amount_(_₹_)',
       'work_status'],
      dtype='object')
Index(['sr._no.', 'work_category', 'work', 'state', 'ida', 'work_description',
       'hon'ble_members_of_parliament', 'constituency', 'image',
       'completion_date', 'amount_disbursed_(_₹_)'],
      dtype='object')


In [8]:
master = (
    recommended
    .merge(
        sanctioned,
        on="work",
        how="left"
    )
    .merge(
        completed,
        on="work",
        how="left"
    )
)

In [15]:
master.columns

Index(['sr._no._x', 'work_category_x', 'work', 'state_x', 'ida_x',
       'hon'ble_members_of_parliament_x', 'constituency_x',
       'work_description_x', 'recommended_date_x',
       'recommended_amount___(_₹_)', 'sanction_date_x', 'sr._no._y',
       'work_category_y', 'state_y', 'ida_y',
       'hon'ble_members_of_parliament_y', 'constituency_y',
       'work_description_y', 'recommended_date_y', 'sanction_date_y',
       'sanction_amount_(_₹_)', 'work_status', 'sr._no.', 'work_category',
       'state', 'ida', 'work_description', 'hon'ble_members_of_parliament',
       'constituency', 'image', 'completion_date', 'amount_disbursed_(_₹_)'],
      dtype='object')

In [12]:
master.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

In [16]:
import spacy

nlp = spacy.load(
    "en_core_web_sm"
)

def get_location(text):

    doc = nlp(str(text))

    locations = []

    for ent in doc.ents:

        if ent.label_ in [
            "GPE",
            "LOC"
        ]:
            locations.append(
                ent.text
            )

    return ", ".join(locations)

master["location"] = (
    master["work_description"]
    .apply(get_location)
)

In [18]:
master['location'].value_counts()

location
                     64461
L.No                   129
Village                 72
Farrukhabad             69
singh                   66
                     ...  
Barheta Panchayat        1
Hamid Molla              1
Chirawa                  1
Chowrastha               1
Chaburta                 1
Name: count, Length: 6190, dtype: int64

In [20]:
from geopy.geocoders import Nominatim

geo = Nominatim(
    user_agent="mplads"
)

def get_lat_lon(row):

    query = (
        f"{row['location']},"
        f"{row['constituency']},"
        f"{row['state']}"
    )

    try:

        loc = geo.geocode(query)

        return (
            loc.latitude,
            loc.longitude
        )

    except:

        return None,None

In [29]:
print(master['sanction_amount_(_₹_)'].value_counts())

sanction_amount_(_₹_)
500000.0     6022
200000.0     3717
300000.0     3503
1000000.0    2426
250000.0     2320
             ... 
2297791         1
495627.0        1
497185.0        1
                1
398955.0        1
Name: count, Length: 13225, dtype: int64


In [32]:
master['amount_disbursed_(_₹_)'] = (
    master['amount_disbursed_(_₹_)']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

master['sanction_amount_(_₹_)'] = (
    master['sanction_amount_(_₹_)']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace('\xa0', '', regex=False)
    .str.strip()
)

master['sanction_amount_(_₹_)'] = pd.to_numeric(
    master['sanction_amount_(_₹_)'],
    errors='coerce'
)

print(master['amount_disbursed_(_₹_)'].dtype)
print(master['sanction_amount_(_₹_)'].dtype)

float64
float64


In [33]:
master[
    "utilization_percent"
] = (
    master["amount_disbursed_(_₹_)"]
    /
    master["sanction_amount_(_₹_)"]
) * 100

In [35]:
date_cols = [
    'recommended_date_x',
    'sanction_date_x',
    'recommended_date_y',
    'sanction_date_y',
    'completion_date'
]

for col in date_cols:
    master[col] = (
        master[col]
        .astype(str)
        .str.strip()
        .replace({
            '': pd.NA,
            ' ': pd.NA,
            '\xa0': pd.NA,
            'nan': pd.NA,
            'NaT': pd.NA
        })
    )

In [36]:
for col in date_cols:
    master[col] = pd.to_datetime(
        master[col],
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )

In [37]:
master["delay_days"] = (master["completion_date"] - master["sanction_date_x"]).dt.days

In [39]:
master["priority_score"] = (
    0.5*master["utilization_percent"]
    -
    0.5*master["delay_days"]
)

In [40]:
def risk(row):

    if row["delay_days"] > 365:
        return "High"

    elif row["delay_days"] > 180:
        return "Medium"

    return "Low"

master["risk"] = (
    master.apply(
        risk,
        axis=1
    )
)

## Till now

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features = [
    "sanctioned_amount",
    "utilization_percent",
    "priority_score"
]

X = master[features]

y = master["risk"]

X_train,X_test,y_train,y_test = (
    train_test_split(
        X,y,
        test_size=0.2,
        random_state=42
    )
)

model = RandomForestClassifier(
    n_estimators=300
)

model.fit(
    X_train,
    y_train
)

In [ ]:
import joblib

joblib.dump(
    model,
    "../models/risk_model.pkl"
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = master[
[
    "sanctioned_amount",
    "utilization_percent"
]]

y = master[
    "delay_days"
]

delay_model = (
    RandomForestRegressor(
        n_estimators=300
    )
)

delay_model.fit(X,y)

In [ ]:
joblib.dump(
    delay_model,
    "../models/delay_model.pkl"
)

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter
)

In [ ]:
from sentence_transformers import (
    SentenceTransformer
)

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
import faiss